In [1]:
##################
### Keagan Moo ###
### 2022/07/XX ###
### Linear Mod ###
##################

# This script runs age-association analysis on the ATAC peaks with a simple linear model, it was used in comparison to the DESeq2 model and GLMM model but does not appear in the manuscript

options(stringsAsFactors = F)
suppressPackageStartupMessages(library(edgeR))
suppressPackageStartupMessages(library(glue))
suppressPackageStartupMessages(library(data.table))
suppressPackageStartupMessages(library(glue))
suppressPackageStartupMessages(library(zoo))
suppressPackageStartupMessages(library(forecast))
suppressPackageStartupMessages(library(parallel))
suppressPackageStartupMessages(library(dplyr))
suppressPackageStartupMessages(library(ggplot2))


In [2]:
feature <- "Type_1"
### LOAD AND SHAPE DATA ###

# Load and organize metadata so that it matches the ARIMA set
# KM # Loads the clustering data, presumably form DESeq non sex specific, in
# KM # a format unfamiliar to me. It appears to be the individual nuclei in rows
# KM # with QA and annotations as features. Some map to the same donor, QA
# KM # includes mitochondrial content.
barcode_md <- file.path(
  "/nfs/turbo/umms-scjp/arushiv/projects/muscle-sn/analyses_hg38/freeze/qc_fusion.bgstringent__k_21",
  paste0("cluster_info_qc.tsv")
) %>% 
  fread(data.table = F)

covariate_md <- file.path(
  "/nfs/turbo/umms-scjp/arushiv/projects/muscle-sn/analyses_hg38/freeze",
  paste0("sample_level_covariates_atac.tsv")
) %>% 
  fread(data.table = F)

covSNG <- data.frame(SNG = covariate_md$SNG.1ST, cluster = substr(covariate_md$coarse_cluster_name, 8, nchar(covariate_md$coarse_cluster_name)),
                     age = covariate_md$age, sex = covariate_md$sex, bmi = covariate_md$bmi, tss_enrichment_sampleCluster = covariate_md$tss_enrichment)

barcode_md$ID <- paste0(barcode_md$SNG.1ST, barcode_md$coarse_cluster_name)
covSNG$ID <- paste0(covSNG$SNG, covSNG$cluster)
covSNG <- covSNG[,!colnames(covSNG) %in% c("SNG", "cluster")]

barcode_md <- merge(barcode_md, covSNG, by = "ID")
barcode_md <- barcode_md[,!colnames(barcode_md) %in% c("ID")]

# KM # Does what it says on the tin. Groups the cellular data by sample
# KM # generates (mutates) a new column for median tss, distinct "fins" values
# KM # and assigns both forms to a new variable. The broad purpose seems to 
# KM # be to make this data conform to a standard "ucar" that I haven't used
metadata.factors <- barcode_md %>%
  group_by(sample) %>%
  mutate(tss_enrich_median = median(tss_enrichment, na.rm = T),
         n_nuclei = n()) %>%
  # cohort:fins gets all columns in this range, including sex, age, etc.
  # we can just get the fins value from the prior dataset but we need to make
  # sure to keep all the columns in between here.
  dplyr::select(sample, batch, cohort, age, sex, bmi, tss_enrich_median, n_nuclei) %>%
  distinct() %>%
  
  # fix wrong sex assignments
  mutate(sex = case_when(sample %in% c("SA02842--564", "SA02842--625") ~ "M",
                         T ~ sex)) %>%
  
  # Keep only FUSION samples
  dplyr::filter(cohort %in% "FUSION") %>%
  
  # Format for model matrices
  mutate(sex = factor(sex, levels = c("M", "F"))) %>%
  mutate(n_nuclei_l10 = log10(n_nuclei)) %>%
  dplyr::filter(!is.na(age)) %>%
  # mutate(age.group = cut(age, seq(20, 80, 5), right = F, ordered_result = T)) %>%
  mutate(age.group = case_when(age <= 40 ~ "young",
                               age > 40 & age <= 64 ~ "midage",
                               age >= 65 ~ "old")) %>%
  mutate(age.group = factor(age.group, levels = c("young", "midage", "old"), ordered = T)) %>%
  mutate(age.group = as.numeric(age.group)) %>%  # yields more readable model matrices
  arrange(age) %>%
  as.data.frame() %>%
  
  # conform to Ucar script
  mutate(sampid = sample,
         batchdate = batch,
         nreps = 1)

# KM # Load in arushi's cell type specific TSS enrichment data so we can use that
TSSDat <- read.table("/nfs/turbo/umms-scjp/arushiv/projects/muscle-sn/analyses_hg38/freeze/cluster-sample-ataqv-stats.txt", header = TRUE)
nameFrame <- data.frame(t(data.frame(strsplit(TSSDat$name, "@"))))
rownames(nameFrame) <- NULL
colnames(nameFrame) <- c("Type","Cohort","sample")
TSSDat$Type <- nameFrame$Type
TSSDat$sample <- nameFrame$sample
TSSDat <- TSSDat[TSSDat$Type == glue("{feature}"),]
TSSDat <- TSSDat %>% dplyr::select(c("sample", "tss_enrichment"))
metadata.factors <- merge(metadata.factors, TSSDat, by = "sample")

# KM # Load in T2D sample data to include in correction
SampleData <- read.table("/nfs/turbo/umms-scjp/arushiv/projects/muscle-sn/analyses_hg38/freeze/batch-sample-info-full.tsv", sep = "\t", header = TRUE)
SampleData <- SampleData[SampleData$sample %in% metadata.factors$sample,]
SampleData <- SampleData %>% dplyr::select(c("sample", "ogtt"))

metadata.factors <- merge(metadata.factors, SampleData, by = "sample")
rownames(metadata.factors) <- metadata.factors$sample

metadata.factors$batch <- as.factor(metadata.factors$batch)



In [3]:
# Load and append peak data in a form matching the ARIMA stuff
peaks_dir <- file.path("/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/Files/ARIMAke/",
                       "arushivData20220415/fusion_150_ccc")

peaks <- file.path(peaks_dir, glue("atac.{feature}.ccc.tsv")) %>%
  fread(data.table = F) %>%
  tibble::column_to_rownames("feature") %>%
  data.matrix()

peaks <- peaks[, colnames(peaks) != "V1"]

Warning message in fread(., data.table = F):
“Detected 286 column names but the data has 287 columns (i.e. invalid file). Added 1 extra default column name for the first column which is guessed to be row names or an index. Use setnames() afterwards if this guess is not correct, or fix the file write command that created the file to create a valid file.”


In [4]:
dim(peaks)
head(peaks)

[1] 203873    285

,12001--510,12002--551,12003--530,12004--530,12006--530,12007--647,12008--530,12009--499,12010--647,12011--499,⋯,32114--499,32115--647,32117--530,32118--551,32119--692,32120--551,32121--510,P14744--530,P14762--564,multiome--3172
chr10:100001774:100002075,0,9,2,3,2,4,1,4,0,3,⋯,0,2,1,2,2,5,1,6,4,11
chr10:100009814:100010115,26,44,27,34,20,16,33,17,0,12,⋯,13,14,27,10,10,22,21,33,18,239
chr10:100019805:100020106,2,3,8,6,0,0,0,0,0,0,⋯,0,0,1,0,2,2,0,2,2,4
chr10:100025760:100026061,5,8,3,11,2,2,3,3,0,4,⋯,0,1,2,0,2,1,0,4,4,11
chr10:100038507:100038808,1,1,0,1,0,0,0,0,0,1,⋯,0,0,0,0,0,2,1,1,1,5
chr10:100043868:100044169,18,37,21,29,6,9,9,10,0,13,⋯,8,9,17,3,4,11,14,28,2,73


In [4]:
# Put everything in the same order and remove samples with missing age
peaks <- peaks[,colnames(peaks) %in% rownames(metadata.factors)]
metadata.factors <- metadata.factors[metadata.factors$sample %in% colnames(peaks),]
peaks <- peaks[,rownames(metadata.factors)]

if (!all(rownames(metadata.factors) == colnames(peaks))) {
  stop("Something wrong (1)")
}

libsizes <- colSums(peaks)

if (all(names(libsizes) == metadata.factors$sample)) {
  message("Adding library sizes to meta data")
  metadata.factors$libsize <- log2(unname(libsizes))
} else {
  stop("Something wrong (2)")
}

Adding library sizes to meta data



In [5]:
### PRE-NORMALIZE COUNTS ###

sampleDepths <- colSums(peaks)
metadata.factors$sDepth <- sampleDepths

# TMM Normalize Data (If we want to use adjusted counts downstream)
fPeaks <- peaks[,colnames(peaks) %in% metadata.factors$sample[metadata.factors$sex == "F"]]
mPeaks <- peaks[,colnames(peaks) %in% metadata.factors$sample[metadata.factors$sex == "M"]]

libsizes <- colSums(peaks)
fLibsizes <- colSums(fPeaks)
mLibsizes <- colSums(mPeaks)

peaksDGE <- DGEList(peaks,lib.size = libsizes)
fPeaksDGE <- DGEList(fPeaks,lib.size = fLibsizes)
mPeaksDGE <- DGEList(mPeaks,lib.size = mLibsizes)

peaksDGE <- calcNormFactors(peaksDGE,method = "TMM")
fPeaksDGE <- calcNormFactors(fPeaksDGE,method = "TMM")
mPeaksDGE <- calcNormFactors(mPeaksDGE,method = "TMM")

peaksNorm <- peaks %*% diag(peaksDGE$samples$norm.factors)
fPeaksNorm <- fPeaks %*% diag(fPeaksDGE$samples$norm.factors)
mPeaksNorm <- mPeaks %*% diag(mPeaksDGE$samples$norm.factors)

colnames(peaksNorm) <- colnames(peaks)
colnames(fPeaksNorm) <- colnames(fPeaks)
colnames(mPeaksNorm) <- colnames(mPeaks)

peaksNormL2 <- log2(1+peaksNorm)
fPeaksNormL2 <- log2(1+fPeaksNorm)
mPeaksNormL2 <- log2(1+mPeaksNorm)

peaksNormMean <- matrix(rep(rowMeans(peaksNorm),ncol(peaksNorm)),nrow = nrow(peaksNorm))
fPeaksNormMean <- matrix(rep(rowMeans(fPeaksNorm),ncol(fPeaksNorm)),nrow = nrow(fPeaksNorm))
mPeaksNormMean <- matrix(rep(rowMeans(mPeaksNorm),ncol(mPeaksNorm)),nrow = nrow(mPeaksNorm))

In [6]:
### LINEAR MODEL WITH COVARIATES ###
fMeta <- metadata.factors[metadata.factors$sex == "F",]
mMeta <- metadata.factors[metadata.factors$sex == "M",]

designNull <- "model.matrix(~bmi + n_nuclei_l10 + batch + tss_enrichment + ogtt + offset(libsize))"
designAge <- "model.matrix(~bmi + n_nuclei_l10 + batch + tss_enrichment + ogtt + offset(libsize) + age)"
designNullParsed <- with(metadata.factors, eval(parse(text=designNull)))
fDesignNullParsed <- with(fMeta, eval(parse(text=designNull)))
mDesignNullParsed <- with(mMeta, eval(parse(text=designNull)))
designAgeParsed <- with(metadata.factors, eval(parse(text=designAge)))
fDesignAgeParsed <- with(fMeta, eval(parse(text=designAge)))
mDesignAgeParsed <- with(mMeta, eval(parse(text=designAge)))

peaksDGE <- estimateDisp(peaksDGE, designNullParsed, robust=T,tagwise = T,trend.method = "locfit")
fPeaksDGE <- estimateDisp(fPeaksDGE, fDesignNullParsed, robust=T,tagwise = T,trend.method = "locfit")
mPeaksDGE <- estimateDisp(mPeaksDGE, mDesignNullParsed, robust=T,tagwise = T,trend.method = "locfit")
#peaksDAGE <- estimateDisp(peaksDGE, designAgeParsed, robust=T,tagwise = T,trend.method = "locfit")
fPeaksDAGE <- estimateDisp(fPeaksDGE, fDesignAgeParsed, robust=T,tagwise = T,trend.method = "locfit")
mPeaksDAGE <- estimateDisp(mPeaksDGE, mDesignAgeParsed, robust=T,tagwise = T,trend.method = "locfit")

designNullGLM <- glmFit(peaksDGE, designNullParsed)
fDesignNullGLM <- glmFit(fPeaksDGE, fDesignNullParsed)
mDesignNullGLM <- glmFit(mPeaksDGE, mDesignNullParsed)
fDesignAgeGLM <- glmFit(fPeaksDAGE, fDesignAgeParsed)
mDesignAgeGLM <- glmFit(mPeaksDAGE, mDesignAgeParsed)

#fDesignAgeLM <- lm(fPeaksDGE$counts ~ metadata.factors$bmi + metadata.factors$tss_enrich_median + metadata.factors$n_nuclei_l10 + metadata.factors$batch)
#mDesignAgeLM <- lm(fPeaksDGE, mDesignAgeParsed)

# Need to specify which coeffecient to test, apparently it defualts to "THE LAST"?
# Ideally we would look into specifying contrast though we've got the age coeff for now
fRes <- topTags(glmLRT(fDesignAgeGLM), n = 200000000)$table
mRes <- topTags(glmLRT(mDesignAgeGLM), n = 200000000)$table

peaksAdjusted <- peaksNormL2 - log2(1+designNullGLM$fitted.values) + peaksNormMean
fPeaksAdjusted <- fPeaksNormL2 - log2(1+fDesignNullGLM$fitted.values) + fPeaksNormMean
mPeaksAdjusted <- mPeaksNormL2 - log2(1+mDesignNullGLM$fitted.values) + mPeaksNormMean

In [7]:
nrow(fRes[fRes$PValue <= 0.05,])
nrow(mRes[mRes$PValue <= 0.05,])
nrow(fRes[fRes$FDR <= 0.1,])
nrow(mRes[mRes$FDR <= 0.1,])

[1] 9082

[1] 13722

[1] 0

[1] 3

In [8]:
saveRDS(list(fRes, mRes) , glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Res/AgeATAC/{feature}_glmoff.rds"))
saveRDS(list(peaksAdjusted, fPeaksAdjusted, mPeaksAdjusted), glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Res/AgeATAC/{feature}_glmoffadj.rds"))

In [3]:
peaky <- readRDS(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Res/AgeATAC/{feature}_glmoffadj.rds"))
peaksAdjusted <- peaky[[1]]
peaksAdjustedm <- peaky[[2]]

In [4]:
linearRes <- readRDS(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Res/AgeATAC/{feature}_glmoff.rds"))

LM <- linearRes[[2]][linearRes[[2]]$FDR <= 0.5,]
LF <- linearRes[[1]][linearRes[[1]]$FDR <= 0.5,]

#Model <- c("GLM", "GLM")
#sexCol <- c("male","female")
#countCol <- c(length(rownames(LM)), 
#                      length(rownames(LF)))
#dfPlot <- data.frame(Model, sexCol, countCol)               
#dfPlot$Model <- factor(dfPlot$Model, levels = c("GLM"))
#pdf(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Plots/AgeATAC/GLMPeakCompare_{feature}.pdf"), width=5, height=5)
#mygg <- ggplot(dfPlot, aes(x = sexCol, y = countCol)) +
#  geom_bar(position = 'dodge', stat = "identity") +
#  xlab("Subject Sex") + ylab("Number of Peaks") + ggtitle(glue("Overlap of Nominally Significant (FDR < 0.1) {feature} Peaks")) + 
#  #scale_fill_manual(values = c("#CC8000","#315CCC","#A9034C","#000000")) +
#  scale_y_continuous(breaks=seq(0,2500,500), limits = c(0,2500)) + theme_bw()
#mygg
#dev.off()

In [6]:
head(LF[LF$logFC < 0,], n = 5)
nrow(LF[LF$logFC < 0,])
nrow(LF[LF$logFC > 0,])
nrow(LM[LM$logFC < 0,])
nrow(LM[LM$logFC > 0,])

,logFC,logCPM,LR,PValue,FDR
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
chr12:118126976:118127277,-0.09720343,1.958492,22.21520,2.437365e-06,0.3347076
chr16:67639769:67640070,-0.06965332,2.422539,21.77712,3.062311e-06,0.3347076
chr5:150024375:150024676,-0.05105416,2.992757,21.53775,3.469311e-06,0.3347076


[1] 3

[1] 0

[1] 0

[1] 0

In [7]:
### PLOT PEAKS OVER AGE TO DETERMINE DISTINCTION ###
# PO # use adjusted data, specifically tss_enrich_median
# Make time series object with average values for each peak at each charted age.

# Replace the sample ID in the dataframe by their age from meta.data factors
# combine rows of the same age by taking the average
# No interpolation, no fancy continuum, maybe z-scale it and then just plot a line
# It looks like this is what the ucar lab do in arima.selections
ages <- with(metadata.factors,setNames(age,sampid))
sexes <- with(metadata.factors,setNames(sex,sampid))
# Get the list of ages in order which appear in male and female samples
mAges <- ages[names(sexes[sexes=="M"])]
fAges <- ages[names(sexes[sexes=="F"])]
# Get the subset of data that fall under that category
mAdj <- peaksAdjusted[,names(sexes[sexes=="M"])]
tmAdj <- t(mAdj)
fAdj <- peaksAdjusted[,names(sexes[sexes=="F"])]
tfAdj <- t(fAdj)
# List out the ages that each sex has access to and their frequencies
mAgeCounts <- as.data.frame(table(mAges),stringsAsFactors = F)
fAgeCounts <- as.data.frame(table(fAges),stringsAsFactors = F)

# average the signal across all samples of the same age. If there is only one sample
# of that age you can simply carry it across
tmAdjUnique <- do.call(rbind,lapply(setNames(as.list(seq_len(nrow(mAgeCounts))),mAgeCounts$mAges),function(n) {
  if (mAgeCounts[n,"Freq"]==1) {
    estage <- tmAdj[mAges==mAgeCounts[n,"mAges"],]
  } else {
    estage <- colMeans(tmAdj[mAges==mAgeCounts[n,"mAges"],])
  }
  return(estage)}))
tfAdjUnique <- do.call(rbind,lapply(setNames(as.list(seq_len(nrow(fAgeCounts))),fAgeCounts$fAges),function(n) {
  if (fAgeCounts[n,"Freq"]==1) {
    estage <- tfAdj[fAges==fAgeCounts[n,"fAges"],]
  } else {
    estage <- colMeans(tfAdj[fAges==fAgeCounts[n,"fAges"],])
  }
  return(estage)}))

# Set type to numeric? I'm not sure if it's necessary
tmAdjUnique <- data.frame(tmAdjUnique)
tfAdjUnique <- data.frame(tfAdjUnique)
mAgesUnique <- as.numeric(mAgeCounts$mAges)
fAgesUnique <- as.numeric(fAgeCounts$fAges)

In [7]:
head(tfAdjUnique)

,chr10.100000195.100000496,chr10.100009814.100010115,chr10.100025760.100026061,chr10.100046058.100046359,chr10.100115078.100115379,chr10.100127715.100128016,chr10.100130818.100131119,chr10.100146903.100147204,chr10.100166760.100167061,chr10.100173107.100173408,⋯,chrX.97295927.97296228,chrX.97367171.97367472,chrX.9786167.9786468,chrX.9874022.9874323,chrX.9885189.9885490,chrX.9899693.9899994,chrX.990099.990400,chrX.99705507.99705808,chrX.9995839.9996140,chrX.9997502.9997803
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
20,11.40794,144.8418,4.771524,47.25653,3.537265,7.870129,28.99999,9.435887,3.525671,4.434406,⋯,5.900816,3.381555,8.372459,7.511963,7.718258,4.140790,82.24102,7.488007,52.84862,4.919451
33,11.17925,145.0005,5.933369,47.93420,4.318916,10.097073,29.52494,10.991440,4.926450,5.406946,⋯,5.207058,2.536585,7.694811,7.591295,7.785326,3.230028,82.72023,6.303972,52.93691,7.012686
47,11.23127,144.8983,3.285992,47.53941,2.510552,8.405178,28.65510,9.911957,2.519806,4.977566,⋯,5.322508,3.990327,7.275139,7.360363,8.330952,4.520064,82.98999,7.530540,52.42742,5.614490
49,11.19923,144.8521,4.384395,47.63309,3.159167,9.057888,29.32356,10.177007,3.491560,4.015571,⋯,4.500948,3.595537,7.068811,7.978887,8.022843,4.439535,82.59765,8.090052,52.48833,6.481837
50,11.58409,144.8964,4.454649,47.80040,2.955356,9.171570,29.23004,9.654025,4.190934,5.572744,⋯,5.366962,3.047035,7.779947,7.514635,8.108011,4.050194,83.67956,6.786452,52.78951,6.281956
51,11.35847,144.7804,4.622484,47.03008,2.652041,9.115706,28.95372,9.863616,3.902708,4.171209,⋯,5.200511,3.544839,7.214175,8.011167,8.647386,4.263379,82.24513,7.781091,52.70980,4.750917


In [ ]:
coords <- "chr5.150024375.150024676"

pdf(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Plots/AgeATAC/PeakLook/{coords}_Female_{feature}.pdf"), width=6, height=6)
mygg <- ggplot(data=tfAdjUnique, aes(x=fAgesUnique, y=chr5.150024375.150024676)) +
  #geom_errorbar(aes(ymin = log2FC_CL, ymax = log2FC_CU)) + 
  geom_line() + geom_point() + theme_bw() +
  xlab("Ages") + ylab("Mean Adjusted Counts")+ ggtitle(coords)
mygg
dev.off()

tfAdjFrame <- t(peaksAdjusted)
tfAdjFrame <- merge(tfAdjFrame, fAges)
colnames(tfAdjFrame) <- c(colnames(tfAdjUnique),"Age")
tmAdjFrame <- t(peaksAdjustedm)
tmAdjFrame <- merge(tmAdjFrame, mAges)
colnames(tmAdjFrame) <- c(colnames(tmAdjUnique),"Age")

pdf(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Plots/AgeATAC/PeakLook/{coords}_Female_{feature}_Scatter.pdf"), width=6, height=6)
mygg <- ggplot(data=tfAdjFrame, aes(x=Age, y=chr5.150024375.150024676)) +
  #geom_errorbar(aes(ymin = log2FC_CL, ymax = log2FC_CU)) + 
  geom_point(alpha = 0.01) + theme_bw() +
  xlab("Ages") + ylab("Adjusted Counts")+ ggtitle(coords)
mygg
dev.off()

pdf 
  2